# MethylSeg Plotting Example

Generate genomic state plots and embeddings from a saved HM450K model.

This example loads a packaged saved model so that the focus stays on the demonstrated operation.

## Input used in this example

MethylSeg accepts tab-delimited `.tsv` or `.tsv.gz` files. WGBS count input
uses `CpG_chrm`, `CpG_beg`, `CpG_end`, `meth`, and `coverage`:

```text
CpG_chrm	CpG_beg	CpG_end	meth	coverage
chr1	10468	10469	7	12
chr1	10470	10471	3	10
```

TCGA/HM450K input uses beta values directly; `probe` is optional:

```text
CpG_chrm	CpG_beg	CpG_end	beta	probe
chr1	10468	10469	0.583	cg00000029
chr1	10470	10471	0.271	cg00000108
```

The complete inputs below are in `data/reference_files/` after running
`methylseg download_data_files`.


## Parameters

Change these explicit values for your own data or output location.

In [ ]:
from pathlib import Path

from methylseg import MethylSegPathway, MethylStateAssignmentMethod, MethylationStates
from methylseg.helper_classes import DATA_DIR

REFERENCE_DIR = DATA_DIR / "reference_files"
OUTPUT_ROOT = Path("methylseg_examples_output")
CHROM = "chr1"

RESOLUTION = "450k"
SAMPLE_NAME = "TCGA-BD-A3EP-01A"
SAMPLE_FILE = REFERENCE_DIR / "TCGA-BD-A3EP-01A_450k.tsv.gz"
OUT_DIR = OUTPUT_ROOT / "methylseg_plotting_example"


## Load a saved model and prepare the sample

In [ ]:
model = MethylSegPathway.get_pretrained_model(
    out_dir=OUT_DIR,
    resolution=RESOLUTION,
)
sample_info, removed_df = MethylSegPathway.prepare_sample_info(
    sample_name=SAMPLE_NAME,
    sample_file=SAMPLE_FILE,
    resolution=RESOLUTION,
    min_coverage=10,
)
sample_info.sample_id

## Generate regions

Run segmentation first so the label and cleaned-region plots have source data.

In [ ]:
regions_chr1 = model.generate_regions(sample_info=sample_info, chrom=CHROM)
regions_chr1.head()

## Genomic label plots

In [ ]:
model.plot_labels(label_source="kmeans", sample_info=sample_info, sample_info_removed=removed_df, chrom=CHROM)
model.plot_labels(label_source="hmm", sample_info=sample_info, sample_info_removed=removed_df, chrom=CHROM)

## Cleaned-region overlays

`get_clean_regions` merges and filters raw calls and writes the chromosome-local metadata used below. With `use_cleaned_regions=True`, the overlay can have fewer or differently bounded regions than the raw segmentation because it reflects those cleaning rules. Run this cell before requesting a cleaned overlay.

In [ ]:
model.get_clean_regions(
    regions_df=regions_chr1, sample_id=sample_info.sample_id, chrom=CHROM
)
model.plot_labels(
    label_source="hmm", sample_info=sample_info, sample_info_removed=removed_df,
    chrom=CHROM, use_cleaned_regions=True, overlay_state="PMD",
    region_start=2_000_000, region_end=4_000_000,
    max_points=120_000,
)

## Embeddings and hexbin density

`hexbin_mincnt` is the minimum number of observations required before a hexagonal bin is drawn. Larger values hide sparse bins and emphasize dense signal; smaller values retain more sparse detail. Set it to `None` to let Matplotlib draw every bin. The remaining key display controls are `region_start`/`region_end` for a genomic viewport, `overlay_state` for a highlighted state, `max_points` for scatter downsampling, and `state_colors` for a custom palette.

In [ ]:
model.plot_embedding(
    label_source="kmeans", sample_info=sample_info, chrom=CHROM, method="pca",
    hexbin=True, hexbin_mincnt=5, include_biplot=True, include_metrics=False,
)

## Outputs

Raw region BED files are written under `OUT_DIR`. Cleaning writes chromosome-local metadata under `OUT_DIR / "clean_regions"`, which is then used by cleaned overlays.